# Focus

This notebooks focus is on getting S&P 500 company data from Wikipedia

In [2]:
print('test to make sure venv is working')

test to make sure venv is working


# Python Imports From Venvironment

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

# Notebook Settings to Help With Development

In [4]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

# Create Folders For Data

In [5]:
RAW_DATA_PATH = Path('../data/raw')
PROCESSED_DATA_PATH = Path('../data/processed')

In [6]:
# Check to see if the directories exist, if not create
if not RAW_DATA_PATH.exists():
    RAW_DATA_PATH.mkdir(parents=True, exist_ok=True)
    print(f"Created directory: {RAW_DATA_PATH}")
else:
    print(f"Directory already exists: {RAW_DATA_PATH}")

if not PROCESSED_DATA_PATH.exists():
    PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)
    print(f"Created directory: {PROCESSED_DATA_PATH}")
else:
    print(f"Directory already exists: {PROCESSED_DATA_PATH}")

Created directory: ../data/raw
Created directory: ../data/processed


# S&P 500 List

Wikipedia keeps a table of all the current S&P 500 companies, so we can just pull that directly instead of typing out tickers by hand like the yfinance notebook does.

Note some of the tickers used in the other notebooks (SPY, QQQ, AGG, GLD, VNQ, VXUS) are ETFs not actual companies, so they won't show up in this table. That's expected, not a bug.

In [8]:
!pip3 install lxml html5lib bs4

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 8.6 MB 5.1 MB/s eta 0:00:01
     |████████████████████████████████| 112 kB 35.0 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [10]:
import requests
from io import StringIO

WIKI_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(WIKI_URL, headers=headers)
tables = pd.read_html(StringIO(response.text))
sp500 = tables[0]

sp500.head()

/Users/danny/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989


# Investigate the Data

In [11]:
sp500.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 503 entries, 0 to 502
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Symbol                 503 non-null    object
 1   Security               503 non-null    object
 2   GICS Sector            503 non-null    object
 3   GICS Sub-Industry      503 non-null    object
 4   Headquarters Location  503 non-null    object
 5   Date added             503 non-null    object
 6   CIK                    503 non-null    int64 
 7   Founded                503 non-null    object
dtypes: int64(1), object(7)
memory usage: 31.6+ KB


We only really need ticker, company name, and sector for now, so narrowing it down to just those columns.

In [12]:
sp500_clean = sp500[['Symbol', 'Security', 'GICS Sector']].copy()
sp500_clean.columns = ['ticker', 'company_name', 'gics_sector']
sp500_clean.head()

,ticker,company_name,gics_sector
0,MMM,3M,Industrials
1,AOS,A. O. Smith,Industrials
2,ABT,Abbott Laboratories,Health Care
3,ABBV,AbbVie,Health Care
4,ACN,Accenture,Information Technology


# Save Outputs

In [13]:
sp500_clean.to_csv(RAW_DATA_PATH / "sp500_constituents.csv", index=False)
print(f"Saved {len(sp500_clean)} rows to {RAW_DATA_PATH / 'sp500_constituents.csv'}")

Saved 503 rows to ../data/raw/sp500_constituents.csv


# Conclusion

## Initial Exploration Summary

In this notebook we pulled the current S&P 500 company list straight from Wikipedia and grabbed the ticker, company name, and sector for each one.

## Main Takeaways

- Wikipedia has a clean table of the current S&P 500 companies, easy to pull with pandas
- This is the current list, not historical, so sector classifications could be outdated for older data
- Some tickers from the other notebooks (like SPY, QQQ) are ETFs and won't match up with this table, need to handle that separately later
- Next step is preprocessing, probably fixing ticker formatting (BRK.B vs BRK-B type stuff) so this can eventually be merged with the price data